# RAG

## Business Setup 
* Insurance Tech Startup
* We have a knowledge base of the company shared drive - knowledge base folder
* task = AI Knowledge Worker

## 'Blunt Instrument' Implementation
* Read names of Employees/Products
* Identify if intent is employee or product related
* add relevant details to prompt


In [11]:
import os
import gradio as gr
from dotenv import load_dotenv
from pathlib import Path
import glob
from openai import OpenAI

In [12]:
load_dotenv(override = True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key.startswith('sk-'):
    print('API Key is valid')
else:
    print('API Key is invalid')




API Key is valid


In [13]:
MODEL = 'gpt-4.1-nano'
openai = OpenAI()

## Write Employee & Product Data in to a Dictionary

In [14]:
knowledge = {}
filenames = glob.glob("knowledge-base/employees/*") # * is a wildcard for all files

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, 'r', encoding='utf-8') as f:
        knowledge[name.lower()] = f.read()

In [15]:
knowledge["lancaster"]

"# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market t

In [16]:
filenames = glob.glob("knowledge-base/products/*")
for filename in filenames:
    name = Path(filename).stem
    with open(filename, 'r', encoding='utf-8') as f:
        knowledge[name.lower()] = f.read()

In [8]:
knowledge.keys()

dict_keys(['kim', 'patel', 'sharma', 'chen', 'spencer', 'foster', 'tran', 'blake', 'zhang', 'thompson', 'adams', 'liu', 'lancaster', 'park', 'greene', 'johnson', 'thomson', "o'brien", 'martinez', 'williams', 'rivera', 'trenton', 'anderson', 'harper', 'rodriguez', 'wilson', 'bishop', 'carter', 'brooks', 'walker', 'rellm', 'claimllm', 'bizllm', 'lifellm', 'healthllm', 'markellm', 'homellm', 'carllm'])

In [17]:
SYSTEM_PREFIX = """
You represent InsureLLM, the Insurance Tech Startup.
You are an expert in answering questions about InsureLLM; its employees and its products.
You are provided with additional context that might be relevant to the users question.
Gove brief accurate answers, if you don't know the answer say so.

Relevant context:

"""

def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace()) # keeps only alphabets and spaces (no punctuation)
    words = text.lower().split()
    relevant_context = []
    for word in words: # iterate through words
        if word in knowledge: # lookup in knowledge base
            relevant_context.append(knowledge[word]) # add to list
    return relevant_context

In [ ]:
# more pythonic - but janky very depedent on keys used so using avery wouldnt work
def get_relevant_context(message):
    text=''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words=text.lower().split()
    return [knowledge[word] for word in words if word in knowledge] # returns a list so can add more to our question

get_relevant_context("Who is lancaster and what is carllm?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [ ]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = 'There is no relevant context for this question'
    else:
        result = f"Here is some additional context that might be relevant to your question: \n\n"
        result += ''.join(relevant_context)
    return result

print(additional_context("Who is lancaster and what is carllm?")) # searching avery still will not work


Here is some additional context that might be relevant to your question: 

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joining Innovate

In [25]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role":"system", "content": system_message}] + history + [{"role":"user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [27]:
view = gr.ChatInterface(chat,title='InsureLLM Search',  type='messages').launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Vectors & Encoding LLMs

There are 2 types of LLM

1. Auto-regressive - It looks back and predicts the next token
2. Auto-encoder - LLMs that produce a specifc output based on a full input

## Auto Encoders

Uses:
* Classification like sentiment analysis
* Vector Embeddings - generating a set on numbers that reflect the meaning and concept of the input in a point of space i.e., 3 number vector could be X,Y,Z in 3 dimensional space

Tokens = inputs, vectors = output. Remember we can put words into a mathematical model

## Vectors

Vectors mathematically represent the meaning of an input 

* They can represent a character, word, an entire document or something abstract.
* Typically scale to thousands of dimensions
* Represent an undertsanding of the input, therefore similar "content" will be close together
* Support vector math -> King - Man + Woman = Queen or London - England + France = Paris

Word2Vec is  one of the first packages that could take a singe word and return a single vector

## RAG & Data Stores

1. We vectorise the users question using code using and Auto-Encoding LLM
2. We then retrieve vectors that are closest to the question i.e., airline rag would retrieve info on tickets to Heathrow and Gatwick if question asked for tickets to london.
3. Question and retrieved info is fed to an Auto-Regressive LLM to generate the response

# Process

1. Read Docs
2. Chunk
3. Vecotrise and Store



# Chunking our Docs

In [28]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [39]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [32]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base= ''

for file_path in files:
    with open(file_path, 'r', encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += '\n\n'

print(f"Total characters in knowledge base: {len(entire_knowledge_base)}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304434


In [ ]:
# tokens across the knowledge base

encoding = tiktoken.encoding_for_model("gpt-4o")
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens in knowledge base: {token_count}")

Total tokens in knowledge base: 63555


In [35]:
folders = glob.glob("knowledge-base/**")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs=loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200) #uses a hierarchy to identify natural chunks
chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")





Total chunks: 413


{'source': 'knowledge-base/products/Rellm.md', 'doc_type': 'products'}

In [38]:
chunks[15]

Document(metadata={'source': 'knowledge-base/products/Bizllm.md', 'doc_type': 'products'}, page_content="### 7. Portfolio Management Dashboard\nComprehensive analytics provide insurers with portfolio-level insights including line-of-business performance, loss ratios, renewal rates, and geographic concentrations. Predictive analytics identify profitable growth opportunities and emerging risks.\n\n### 8. Agent and Broker Portal\nDedicated portals enable distribution partners to quote, bind, and service commercial policies efficiently. Real-time commission tracking, marketing materials, and training resources support productive agency relationships.\n\n### 9. Claims Management Integration\nSeamless connection with claims systems enables fast incident reporting, reserve establishment, and loss tracking. AI-powered tools identify subrogation opportunities and predict claim severity for better reserves.\n\n## Pricing\n\nBizllm's pricing structure reflects the complexity and value of commerci

# Embedding our chunks and storing in Chroma

## Popular Embedding Models

Note different embedding models perform better for different use cases

* OpenAI - text embedding 3 small & large
* Google - gemini-embedding-001
* HF Sentence Transformers - all-MiniLM-L6-V2

## Popular Vector Stores - mainstream DBs now take vectors natively

!! The choice now falls on security, cost, performance, backups and resilience - infrastructure!!

* Open-Source - Chroma, Qdrant, FAISS (in-memory rather than an actual DB)
* Paid - Pinecone, Weaviate
* Mainstream Databases - Mongo, Postgres, Elastic

In [41]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory = db_name, embedding_function=embeddings).delete_collection()

vectorstore=Chroma.from_documents(documents=chunks, embedding=embeddings,persist_directory=db_name)

print(f"Vector store created in {db_name} with {vectorstore._collection.count()} documents")


Vector store created in vector_db with 413 documents


In [ ]:
# looking at the vector store

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store") 
# dimensions is a feature of the embedding model as DBs can take vectors of any size

There are 413 vectors with 384 dimensions in the vector store


# Visualise with TSNe

In [45]:
result = collection.get(include = ['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [48]:
# dimensionality reduction to reduce from 314 to 2d using TSNe

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# 2d scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t,d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualisation',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)

)

fig.show()

# Day 3 - Full E2E Pipeline

In [50]:
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [52]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=db_name, embedding_function = embeddings)

## Setup LangChain Retriver & LLM

In [62]:

retriever = vectorstore.as_retriever() # as_retriever packages the vectorstore as a langchain object that we can call invoke on
llm = ChatOpenAI(temperature=0, model_name='gpt-4.1-mini')

In [ ]:
retriever.invoke("Who is Avery?") # returns langchain docs closest to that question
# questiuon gets vectorised using the retrievers encoder

[Document(id='3f625c7c-7166-4606-868d-6e9e5952570c', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

# 

In [63]:
SYSTEM_PROMPT_TEMPLATE =""" 
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [64]:
answer_question("Who is Avery?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has played a key role in establishing Insurellm as a leading provider in the insurance technology sector. Avery is recognized for her innovative leadership, risk management expertise, and commitment to diversity and community engagement. She has a strong track record of driving product innovation, improving company culture, and leading the company through various market challenges. Avery is based in San Francisco, California, and currently earns a salary of $225,000.'

In [65]:
gr.ChatInterface(answer_question).launch()

/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning:

The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.



* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


# Extra RAG demo for DAY 3 where you can test rag and see context retrieved!

## Productionising RAG

There are two python modules in the implementation directory.

1. Ingest.py - reads documents in the knowledge base, chunks, vectorises then stores in Chroma

2. answer.py - fetch_context and answer questions functions (our retriever and llm)

In the root, we have app.py which is our gradio app

### Pros

1. Rapid Time to Market
2. Gives an LLM vast expertise
3. Context length savings - selecting only the most relevant
4. Minimises poisoning of context with irrelevant info

### Cons

1. It's a hack - Transformers have powerful attention mechanisms however today we have too much info.
2. "It's empirical" trial and error as different RAG techniques are better for different use cases.
3. They are not fool proof - always a chance of the unexpected "I dont know"

## Evaluations

Chunking strategy can have a significant impact on performance so how can we turn it inot a science? EVALS!

1. Curate a test set! Example question set with the right context identified and reference answers provided (make sure this adpats as you find problems)
2. Measure retrieval (how good we retrieve the right context) - MRR (how far up was the first hit), nDCG (distribution of all relevant chunks/hits), Recall (Proportion of relevant chunks in top K) and Precision @ K (proportion of top K chunks that are relevant)
3. Measure answers using gold standards, LLM and experts as judges against criteria like accuracy, completeness, and relevance.



In [70]:
from evaluation import test # see evaluation dictionary

tests=test.load_tests()

example = tests[0]

print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [71]:
from collections import Counter

count = Counter([t.category for t in tests])
count


Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [73]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [ ]:
evaluate_retrieval(example)

## Gradio

cd week5

uv run evaluator.py 

to run gradio Interface. 

Top Tip: If logic is already coded, LLMs can become extremely efficient and powerful at generating UI's


## Interpreting Scores

An MRR or nCDG of 1 isn't always needed (as well as well being very difficult to achieve). Work with stakeholders to find the right threshold.


## Chunking Strategies

Manipulate ingest.py to experiemnt with how different chunks effect retrieval.

When testing make sure to keep things equal i.e., tokens in the conetxt window so 5 chunks of 1000 vs 10 chunks of 500 vs 3 chunks of 1600

* Retrieving smaller but more chunks for context can sometime help with accuracy

# Day 5 - Advanced Rag

## Techniques for Experimentation

1. **Chunking R&D** - Experiment with chunking strategy
2. **Encoder R&D** - Select the best encoder model based on a test set. Also when dealing with images, vectorise a piece of text that best describes it for better. Use knowledge transformer (python) to convert docs into formats better for vectorisation
3. **Prompt Engineering** - General Content, current info, relevany context, history
4. **Doc Preprocessing** - LLM to rewrite knowledge most suitable for querying i.e., turning a table into key-value pairs. Semantic Chunking - splitting up documents by meaning rather than fixed thresholds.
5. **Query Rewrite** - LLM rewrites users query to optimise it for retrieval.
6. **Query Expansion** - LLM writes multiple similar queries (casting a wider net)
7. **Re-Ranking** - LLM to sub-select and reorder from RAG Results.
8. **Hierarchical** - Summarise at multiple levels i.e., hit the compressed summary data and retrieve the related fine grain information
9. **GraphRAG** - retrieve content closely related to similar documents. Chunks can be related i.e., a chunk about an employee and a chunk about their boss or their department and then that departments products or payroll. 1/2/3 hop relationships
10. **Agentic RAG** - use agents for retrieval, combining with Memory and Tools such as SQL, or string lookup for local files etc.

Why RAG isn't dead

* Context Windows may be larger however remmoving irrelevant context is always a priority
* LLM's perform better when context window is at 40% - sharp degradation beyond this due to "Lost-in-the-Middle" Effect. LLM pays attention to info at the start and end of its prompt but starts to fail with info in the middle.
* Prevents Context Drift - When an LLM processes massive amounts of text, the system instructions (e.g., "Adopt a professional tone and format the output as a table") get diluted by the background data. The model can experience "context drift," losing track of its original goal and leading to hallucinations.
* Governance - if you dump everything in the model then you canot control what info is leveraged for a specific user.

In [81]:
! pip install tdqm


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [96]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm # creates progress bar for iterations
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

load_dotenv(override=True)

MODEL = "gpt-5.6-luna"

DB_NAME = "preprocessed_db"
collection_name = "docs"

embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai= OpenAI()


In [99]:
# create pydantic object inspired by langchains document

class Result(BaseModel):
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    header: str = Field(description=" A brief heading for this chunk, typically a few words that are likely to be surfaced in a query")
    summary: str = Field(description="A couple of lines summarising this chunk to answer common queries")
    original_text: str = Field(description="The full unadulterated text from the provided document")
    
    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.header + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)

class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

In [89]:
def fetch_documents():
    """Homemade version of Langchain DirectoryLoader """
    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type= folder.name
        for file in folder.rglob("*.md"):
            with open(file, 'r', encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})
    
    print(f"loaded {len(documents)} documents")
    return documents

In [91]:
documents= fetch_documents()

loaded 76 documents


In [92]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [93]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n#

In [100]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks= Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [101]:
process_document(documents[0])

/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{"chunks...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])



[Result(page_content='Rellm Product Overview and AI Analytics\n\nRellm is Insurellm’s AI-powered enterprise reinsurance platform. It supports risk management, decision-making, operational efficiency, integrations, and real-time predictive analytics.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, en

In [103]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

chunks = create_chunks(documents)

print(len(chunks))

  0%|          | 0/76 [00:00<?, ?it/s]/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning:

Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{"chunks...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])

100%|██████████| 76/76 [47:12<00:00, 37.28s/it]   

582


### Processing Optimisation 


In the python module version, I sneakily use the multi-processing Pool to run this in parallel, **See ingest.py in pro_implementation**


### Step 3 - Embed the chunks

In [109]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts= [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vetcorstore created with {collection.count()} documents")

In [110]:
create_embeddings(chunks)

Vetcorstore created with 582 documents


In [112]:
chroma=PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata['type'] for metadata in metadatas]
colors = [["blue", 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types] 

In [116]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visulaisation',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [120]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='3D Chroma Vector Store Visulaisation',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## Re-Ranking & Query Expansion

In [121]:
class RankOrder(BaseModel):
    order: list[int] = Field(description="The order of relevance of chunks, from most to least relevant, by chunk id number")

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt += f"The user has asked the following question: \n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most to least relevant. Include all the chunk ids you are provided with, reranked"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += "reply only with the list of ranked chunk ids,nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content":user_prompt},
    ]
    response = completion(model=MODEL, messages = messages, response_format=RankOrder) # calling llm
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply)
    print(order)
    return [chunks[i-1] for i in order]

In [ ]:
RETRIEVAL_K = 10

# manual retrieval 

def fetch_context_unranked(question):
    # langchain has its own retriever but model and config is unclear - here we can do our own thing
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    # .data[0] accesses the first result of the returned embedding data
    # .embedding extracts the actual vector of numbers 
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    # collection.query searches our chroma db using the query vectors
    # we use [query] as api accepts a list of query embeddings
    chunks=[]
    for result in zip(results["documents"][0], results["metadatas"][0]):
        # "documents = retrieved text chunks and metadatas = associated metatadat."
        # we use [0] as vector db response is nested so it can handle multiple queries at once 
        # but we only used 1 question so [0] selects the first result from the search
        chunks.append(Result(page_content=result[0], metadata=result[1]))
        # adds each chunk tto the list using structured output from our Result class
    return chunks
